# ICORE — choose one problem or compare all 393

This notebook uses **arithmetic mean time in milliseconds**, never silently drops failed solves,
and shows faster, slower, tied, and incomparable cases separately.

To learn how to build your own model, start with [Tutorial 1](tutorials/01_scalar_regulator.ipynb).
All stored coefficient examples are in `problems/`. The default comparison methods are
ICORE C, ICORE C++, HPIPM BALANCE, and QOCO.

> **Development transparency:** Substantial portions of ICORE's implementation and documentation were generated and revised with GPT under human direction. Not all details have undergone independent human review. See the [development and review statement](README.md#development-transparency-and-review-status).


In [ ]:
# Use a fresh kernel: limit threaded BLAS before importing NumPy/pandas.
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[name] = "1"

from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "icore" / "api.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from example_tools.catalog import list_problems, load_problem
from example_tools.runner import prepare, run_problem, run_all, mean_table
from example_tools.display import show_catalog, show_problem, show_decisions, show_comparisons, export_tables
from example_tools.trace import replay

METHODS = ("ICORE C", "ICORE C++", "HPIPM", "QOCO")
TOLERANCE = 1e-8
N = 5                          # Number of cold solves per method, not IPM iterations.
prepared = prepare(METHODS)     # Native compilation is outside measured solve time.
display(pd.DataFrame(prepared["solvers"]).drop(columns=["compiler"], errors="ignore"))

## 1. Browse the catalog
Filter by a name fragment or family; these filters only change the display.
393 coefficient instances include synthetic tests and model updates, not 393 independent applications.

In [ ]:
show_catalog()
FILTER = "LIPM"                 # Set to "" to show every problem.
entries = list_problems(contains=FILTER)
display(pd.DataFrame(entries)[["name", "family", "N", "nx", "nu", "n_inequalities"]])

## 2. Select, display, and solve one problem

In [ ]:
PROBLEM_NAME = "LIPMWALK0"
problem = load_problem(PROBLEM_NAME)
formulation = show_problem(problem)
selected = run_problem(problem, repetitions=N, methods=METHODS,
                       tolerance=TOLERANCE, prepared=prepared)
display(mean_table(selected["rows"], repetitions=N))

## 3. Inspect the actual execution
Basic counters come from the measured production solve. Set `DETAILED=True` for one extra,
untimed diagnostic solve. Its output must match production before its trace is used.

In [ ]:
BACKEND = "cpp"
DETAILED = False
method = "ICORE C++" if BACKEND == "cpp" else "ICORE C"
trace = None
if DETAILED:
    trace = replay(problem, backend=BACKEND, tolerance=TOLERANCE,
                   production_result=selected["latest_solutions"].get(method))
show_decisions(problem, selected, backend=BACKEND, trace=trace)

## 4. All mode
Set `RUN_ALL=True` to run **every catalog problem** with each method `ALL_N` times.
Each repetition uses a fresh workspace and a cold solve. Compilation, model conversion,
correctness audits, and reporting are excluded from solve timing.

Results are saved after each problem. `RESUME=True` resumes only a compatible saved run.
Setup and solve means are reported separately; `total_ms` means setup + solve, not the entire application.
The detailed diagnostic replay above is never part of All mode.

In [ ]:
RUN_ALL = False
ALL_N = 5
OUTPUT = ROOT / "runs" / f"all393_N{ALL_N}"
RESUME = False
all_results = None

def progress(state):
    if state["completed"] % 25 == 0 or state["complete"]:
        print(f"{state['completed']}/{state['total']} saved; last: {state['last']}")

if RUN_ALL:
    all_results = run_all(repetitions=ALL_N, methods=METHODS, tolerance=TOLERANCE,
                          prepared=prepared, output_dir=OUTPUT, resume=RESUME, progress=progress)
    print("Completed:", all_results["complete"])
else:
    print("Set RUN_ALL=True to run all 393 problems.")

## 5. Faster and slower tables
By default the reference is the **fastest valid requested rival** for each problem.
Use `RIVALS=("HPIPM",)` or `("QOCO",)` to compare with one rival.
A method must pass all N/N accuracy checks; failures and unsupported cases do not become speed wins.
Partial rival coverage is explicitly labeled. A faster mean is not a statistical significance claim.

In [ ]:
ICORE_METHOD = "ICORE C++"      # Or "ICORE C".
RIVALS = ("HPIPM", "QOCO")
METRIC = "solve_ms"             # Or "setup_ms", "total_ms".

results_to_show = all_results if all_results is not None else selected
N_to_show = results_to_show["repetitions"]
tables = show_comparisons(results_to_show["rows"], repetitions=N_to_show,
                          icore_solver=ICORE_METHOD, rivals=RIVALS, metric=METRIC)
if all_results is not None:
    export_tables(tables, OUTPUT / "tables")

## Notes
- All numeric tolerances and models remain unchanged by this reporting layer.
- The HPIPM QP adapter explicitly skips generic SOCs; the model is not approximated.
- Model assembly, first compilation, setup, solver time, and complete application latency are different quantities.
- Use larger N and repeated sessions for performance decisions. Hardware and CPU load matter.
- Runs and build caches are ignored by Git; they are not part of the source repository.
- See [benchmark details](docs/BENCHMARKING.md) and [the API contract](docs/API.md).